### **Import Library**

In [1]:
import pandas as pd
import numpy as np

### **Setting**

In [2]:
# Prevent wrapping rows onto new lines
pd.set_option('display.width', 1000)
pd.set_option('display.max_columns', None)

# Format scientific notation into standard numbers with commas
pd.set_option('display.float_format', lambda x: f'{x:,.0f}')

### **Read File**

In [3]:
employment = pd.read_csv("../Data/Gross Value Added and Employment.csv")
tax_and_subsidies = pd.read_csv("../Data/Gross Value Added, Taxes and Subsidies.csv")

### **List out their features**

In [4]:
# Display column names and data types
print("\nGross Value Added and Employment")
print(employment.dtypes)

print("\nGross Value Added, Taxes and Subsidies")
print(tax_and_subsidies.dtypes)


Gross Value Added and Employment
Statistic Label           str
Year                    int64
NACE Rev. 2 Sector        str
NUTS 2 Region             str
UNIT                      str
VALUE                 float64
dtype: object

Gross Value Added, Taxes and Subsidies
Statistic Label        str
Year                 int64
NUTS 2 Region          str
UNIT                   str
VALUE              float64
dtype: object


### **Try to find out the common between three db**

In [5]:
# To inspect the exact unique entries from your loaded DataFrames
for name, df in [("Employment", employment), ("Taxes & Subsidies", tax_and_subsidies)]:
    print(f"{name}")
    print("Statistic Labels:", df["Statistic Label"].unique().tolist())
    print("Years:", sorted(df["Year"].unique().tolist()))
    print("Units:", df["UNIT"].unique().tolist())
    print("\n")

Employment
Statistic Labels: ['Gross Value Added (GVA) at Basic Prices', 'Total Persons at Work', 'Employees at Work', 'Self-Employed at Work', 'Hours Worked']
Years: [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Units: ['Euro Million', 'Number', 'Thousand']


Taxes & Subsidies
Statistic Labels: ['Compensation of Employees', 'Gross Value Added (GVA) at Factor Cost', 'Non-Product Taxes', 'Non-Product Subsidies', 'Gross Value Added (GVA) at Basic Prices', 'Gross Value Added (GVA) at Basic Prices per Person', 'Product Taxes', 'Product Subsidies', 'Gross Domestic Product in Current Market Prices', 'Gross Domestic Product (GDP) per Person', 'Population', 'Apparent Productivity', 'Persons at Work', 'Index of GVA at Basic Prices per Person (State=100)', 'Hours Worked']
Years: [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 

#### We can notice the common year range for three db is from 2000 to 2024

### **Cleaning Dataframe**

### Merge the feature of unit and value

In [6]:
# Define unit multipliers
unit_multipliers = {
    'Euro Million': 1_000_000,
    'Persons Thousand': 1_000,
    'Thousand': 1_000,
    'Euro': 1,
    'Number': 1_000
}

# Function to scale value and remove unit col
def normalize_value_and_drop_unit(df):
    # Map the unit to its numeric multiplier
    multipliers = df['UNIT'].map(unit_multipliers)
    
    df['VALUE'] = df['VALUE'] * multipliers
    
    # Drop the unit col
    df = df.drop(columns=['UNIT'])
    
    return df

# Apply to df
employment = normalize_value_and_drop_unit(employment)
tax_and_subsidies = normalize_value_and_drop_unit(tax_and_subsidies)


print("\nGross Value Added and Employment")
print(employment.head())

print("\nGross Value Added, Taxes and Subsidies")
print(tax_and_subsidies.head())


Gross Value Added and Employment
                           Statistic Label  Year                 NACE Rev. 2 Sector         NUTS 2 Region         VALUE
0  Gross Value Added (GVA) at Basic Prices  2000  Agriculture, Forestry and Fishing  Northern and Western   575,940,000
1  Gross Value Added (GVA) at Basic Prices  2000  Agriculture, Forestry and Fishing              Southern 1,519,270,000
2  Gross Value Added (GVA) at Basic Prices  2000  Agriculture, Forestry and Fishing   Eastern and Midland   622,520,000
3  Gross Value Added (GVA) at Basic Prices  2000  Agriculture, Forestry and Fishing               Ireland 2,717,730,000
4  Gross Value Added (GVA) at Basic Prices  2000      Industry (excl. Construction)  Northern and Western 3,504,610,000

Gross Value Added, Taxes and Subsidies
             Statistic Label  Year         NUTS 2 Region          VALUE
0  Compensation of Employees  2000  Northern and Western  5,933,751,057
1  Compensation of Employees  2000              Southern 13,39

### Check Missing Value

In [7]:
print("\nGross Value Added and Employment")
print(employment.isnull().sum())

print("\nGross Value Added, Taxes and Subsidies")
print(tax_and_subsidies.isnull().sum())


Gross Value Added and Employment
Statistic Label        0
Year                   0
NACE Rev. 2 Sector     0
NUTS 2 Region          0
VALUE                 48
dtype: int64

Gross Value Added, Taxes and Subsidies
Statistic Label    0
Year               0
NUTS 2 Region      0
VALUE              4
dtype: int64


### Remove Missing Value

In [8]:
# Remove Missing Values
employment = employment.dropna(subset=['VALUE']).reset_index(drop=True)
tax_and_subsidies = tax_and_subsidies.dropna(subset=['VALUE']).reset_index(drop=True)

print("\nGross Value Added and Employment")
print(employment.isnull().sum())

print("\nGross Value Added, Taxes and Subsidies")
print(tax_and_subsidies.isnull().sum())


Gross Value Added and Employment
Statistic Label       0
Year                  0
NACE Rev. 2 Sector    0
NUTS 2 Region         0
VALUE                 0
dtype: int64

Gross Value Added, Taxes and Subsidies
Statistic Label    0
Year               0
NUTS 2 Region      0
VALUE              0
dtype: int64


### **Pivoting Dataframe**

### List the label

In [9]:
print("Gross Value Added and Employment")
print("Statistic Labels:")
print(employment['Statistic Label'].unique())

print("\n")

print("Gross Value Added, Taxes and Subsidies")
print("Statistic Labels:")
print(tax_and_subsidies['Statistic Label'].unique())

Gross Value Added and Employment
Statistic Labels:
<StringArray>
['Gross Value Added (GVA) at Basic Prices', 'Total Persons at Work', 'Employees at Work', 'Self-Employed at Work', 'Hours Worked']
Length: 5, dtype: str


Gross Value Added, Taxes and Subsidies
Statistic Labels:
<StringArray>
['Compensation of Employees', 'Gross Value Added (GVA) at Factor Cost', 'Non-Product Taxes', 'Non-Product Subsidies', 'Gross Value Added (GVA) at Basic Prices', 'Gross Value Added (GVA) at Basic Prices per Person', 'Product Taxes', 'Product Subsidies', 'Gross Domestic Product in Current Market Prices', 'Gross Domestic Product (GDP) per Person', 'Population', 'Apparent Productivity', 'Persons at Work', 'Index of GVA at Basic Prices per Person (State=100)', 'Hours Worked']
Length: 15, dtype: str


### Pivot 

In [10]:
# Pivot Employment DataFrame
emp_wide = employment.pivot_table(
    index=['Year', 'NUTS 2 Region', 'NACE Rev. 2 Sector'],
    columns='Statistic Label',
    values='VALUE',
    aggfunc='first'
).reset_index()

# Clean up column index name
emp_wide.columns.name = None


# Pivot Tax and Subsidies DataFrame
tax_wide = tax_and_subsidies.pivot_table(
    index=['Year', 'NUTS 2 Region'],
    columns='Statistic Label',
    values='VALUE',
    aggfunc='first'
).reset_index()

# Clean up column index name
tax_wide.columns.name = None

### Display and check missing value

In [11]:
print("Sector-Level Table (emp_wide)")
print("First 5 rows:")
print(emp_wide.head())
print("\nMissing Values:")
print(emp_wide.isnull().sum())

print("\n" + "="*80 + "\n")

print("Regional Totals Table (tax_wide)")
print("First 5 rows:")
print(tax_wide.head())
print("\nMissing Values:")
print(tax_wide.isnull().sum())

Sector-Level Table (emp_wide)
First 5 rows:
   Year        NUTS 2 Region                               NACE Rev. 2 Sector  Employees at Work  Gross Value Added (GVA) at Basic Prices  Hours Worked  Self-Employed at Work  Total Persons at Work
0  2000  Eastern and Midland                Agriculture, Forestry and Fishing              7,840                              622,520,000    72,183,125                 17,480                 25,320
1  2000  Eastern and Midland                                      All Sectors            772,840                           55,448,600,000 1,642,808,113                118,020                890,860
2  2000  Eastern and Midland           Arts, Entertainment and Other Services             33,790                            1,304,130,000    72,223,471                  8,570                 42,360
3  2000  Eastern and Midland                                     Construction             47,980                            3,641,600,000   134,119,562             

### Remove missing value

In [12]:
emp_wide = emp_wide.dropna().reset_index(drop=True)
tax_wide = tax_wide.dropna().reset_index(drop=True)

print("Missing Values")
print(emp_wide.isnull().sum())
print(tax_wide.isnull().sum())

Missing Values
Year                                       0
NUTS 2 Region                              0
NACE Rev. 2 Sector                         0
Employees at Work                          0
Gross Value Added (GVA) at Basic Prices    0
Hours Worked                               0
Self-Employed at Work                      0
Total Persons at Work                      0
dtype: int64
Year                                                   0
NUTS 2 Region                                          0
Apparent Productivity                                  0
Compensation of Employees                              0
Gross Domestic Product (GDP) per Person                0
Gross Domestic Product in Current Market Prices        0
Gross Value Added (GVA) at Basic Prices                0
Gross Value Added (GVA) at Basic Prices per Person     0
Gross Value Added (GVA) at Factor Cost                 0
Hours Worked                                           0
Index of GVA at Basic Prices per Person (S

### **Export Cleaned CSV**

In [13]:
emp_wide.to_csv('employment_by_sector_clean.csv', index=False)
tax_wide.to_csv('tax_and_gdp_by_region_clean.csv', index=False)